In [ ]:
import numpy as np
import pandas as pd
import os, sys

from collections import Counter
import matplotlib.pyplot as plt
import pickle
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

/opt/anaconda3/envs/transformers/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
from transformers import AutoTokenizer

In [ ]:
## path to the downloaded gutenberg corpus
path_gutenberg = os.path.join(os.pardir,os.pardir,'gutenberg')

In [ ]:
path_gutenberg

'../../gutenberg'

In [ ]:
src_dir = '/Users/nikitaparulekar/GitRepos/gutenberg-analysis/src'
sys.path.append(src_dir)
from data_io import get_book

In [ ]:
sys.path.append(os.path.join(path_gutenberg, 'src'))
from metaquery import meta_query
mq = meta_query(path=os.path.join(path_gutenberg, 'metadata', 'metadata.csv'))


In [ ]:
#mq.df['author'].unique()

In [ ]:
# get the updated csv
data_path = os.path.join(os.pardir, 'sample_dataset')
train_path = os.path.join(data_path, 'final_train.csv')
test_path = os.path.join(data_path, 'final_test.csv')
val_path = os.path.join(data_path, 'final_val.csv')

In [ ]:
train_meta_df = pd.read_csv(train_path)
test_meta_df = pd.read_csv(test_path)
val_meta_df = pd.read_csv(val_path)

In [ ]:
val_meta_df['author'].nunique(),test_meta_df['author'].nunique(),train_meta_df['author'].nunique()

(80, 80, 80)

In [ ]:
test_meta_df['author'].value_counts().unique()

array([3])

In [ ]:
# def split_test_validation(test_meta_df, val_samples_per_author=3):
#     author_list = test_meta_df['author'].unique()
#     val_indices = []

#     for author in author_list:
#         # Get indices for the current author
#         author_indices = test_meta_df[test_meta_df['author'] == author].index

#         # Sample validation indices for this author
#         author_val_indices = np.random.choice(author_indices, val_samples_per_author, replace=False)
#         val_indices.extend(author_val_indices)

#     # Create validation dataframe
#     val_df = test_meta_df.loc[val_indices]

#     # Remove validation samples from test dataframe
#     updated_test_df = test_meta_df.drop(val_indices)

#     return updated_test_df, val_df

In [ ]:
# test_meta_df, val_meta_df = split_test_validation(test_meta_df)

In [ ]:
# test_meta_df.loc[test_meta_df['author'] == 'Doyle, Arthur Conan']

,Unnamed: 0,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects
4,712,PG108,The Return of Sherlock Holmes,"Doyle, Arthur Conan",1859.0,1930.0,['en'],3348,"{'Detective and mystery stories, English', 'Ho..."
97,15728,PG290,The Stark Munro Letters: Being series of twelv...,"Doyle, Arthur Conan",1859.0,1930.0,['en'],265,{'Epistolary fiction'}
301,40867,PG58574,Index of the Project Gutenberg Works of Arthur...,"Doyle, Arthur Conan",1859.0,1930.0,['en'],310,{'Indexes'}


In [ ]:
# val_meta_df.loc[val_meta_df['author'] == 'Doyle, Arthur Conan']

,Unnamed: 0,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects
98,16089,PG294,"The Captain of the Polestar, and Other Tales","Doyle, Arthur Conan",1859.0,1930.0,['en'],508,"{'Short stories, English', 'Great Britain -- S..."
466,56112,PG903,The White Company,"Doyle, Arthur Conan",1859.0,1930.0,['en'],1094,"{'Archers -- Fiction', 'Knights and knighthood..."
353,46299,PG65045,"The British Campaign in France and Flanders, 1917","Doyle, Arthur Conan",1859.0,1930.0,['en'],164,"{'World War, 1914-1918 -- Campaigns -- Western..."


In [ ]:
# test_meta_df['author'].value_counts()

author
Doyle, Arthur Conan                         3
Shakespeare, William                        3
A. L. O. E.                                 3
James, G. P. R. (George Payne Rainsford)    3
Cannon, Richard                             3
                                           ..
Kipling, Rudyard                            3
Davis, Richard Harding                      3
Wells, H. G. (Herbert George)               3
London, Jack                                3
Lytton, Edward Bulwer Lytton, Baron         3
Name: count, Length: 80, dtype: int64

In [ ]:
# val_meta_df['author'].value_counts()

author
Jacobs, W. W. (William Wymark)         3
London, Jack                           3
Harper, Charles G. (Charles George)    3
Miller, Alex. McVeigh, Mrs.            3
Leinster, Murray                       3
                                      ..
Stevenson, Robert Louis                3
Sinclair, Upton                        3
Tolstoy, Leo, graf                     3
Chesterton, G. K. (Gilbert Keith)      3
Lytton, Edward Bulwer Lytton, Baron    3
Name: count, Length: 80, dtype: int64

In [ ]:
# val_path = os.path.join(os.pardir,'sample_dataset', 'final_val.csv')
# val_meta_df.to_csv(val_path)
# test_path = os.path.join(os.pardir,'sample_dataset', 'final_test.csv')
# test_meta_df.to_csv(test_path)


In [ ]:
train_meta_df.head()

,Unnamed: 0,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects
0,2439,PG12810,"Uncle Sam's Boys with Pershing's Troops: Or, D...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],78,"{'World War, 1914-1918 -- Juvenile fiction', '..."
1,2446,PG12819,"Dick Prescott's Second Year at West Point: Or,...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],94,{'United States Military Academy -- Juvenile f...
2,25920,PG40605,"The Motor Boat Club at Nantucket; or, The Myst...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],189,"{'Motorboats -- Juvenile fiction', 'Nantucket ..."
3,55435,PG8153,"The Young Engineers in Arizona; or, Laying Tra...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],190,"{'Civil engineers -- Fiction', 'Arizona -- Fic..."
4,32899,PG48863,"The Motor Boat Club off Long Island; or, A Dar...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],85,"{'Motorboats -- Juvenile fiction', 'Long Islan..."


In [ ]:
# get the full test for each id-author (how to store?)
# tokenize the text
# pretrain transformer with txt classification

In [ ]:
train_meta_df['author'].nunique()

80

In [ ]:
# train_meta_df['author'].unique()

In [ ]:
num_authors = train_meta_df['author'].nunique()

# Tokenization

In [ ]:
# tokenize the books
# get 20 chunks of 512 tokens each, ~uniformly through each book

In [ ]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
# tokenizer

In [ ]:
# encoded_text = tokenizer("I am hungry, and want 26 Birria tacos ")

In [ ]:
# encoded_text["input_ids"][0]

In [ ]:
# tokens = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
# print(tokens)

In [ ]:
# print(tokenizer.convert_tokens_to_string(tokens))

In [ ]:
# train_meta_df.head()

In [ ]:
# def tokenize(batch):
#     '''keep last n tokens, ignore '''
#     return tokenizer(batch["text"], padding=True, truncation=True,truncation_side="left")

In [ ]:
# train_meta_df['author'].unique()

In [ ]:
# book_1 = get_book('PG12810', level='text')

In [ ]:
# book_1[-512:]

# Approach 1: Chunking and Aggregating

In [ ]:
# def get_token_samples(num_samples_per_book,meta_df):
#     ''''Extract num_samples_per_book of max size truncated from left'''
#     # for each author
#     for author in meta_df['author'].unique():
#         author_df = meta_df.loc[meta_df['author'] == author]
#         # for each book id , get full book
#         for id in author_df['id'].unique():
#             book_str = get_book(id, level='text')
#             # tokenize full text (no truncation)

#             # for sample in num_samples_per_book
#             for sample_num in range(num_samples_per_book):
#                 # take last 512 * num_samples_per_book of token_list
#                 # partition that into num_samples_per_book array
#                 #store in a dict of author:nested array of all samples

In [ ]:
# dummy_meta_df = train_meta_df.loc[(train_meta_df['id'] == 'PG63902') | (train_meta_df['id'] == 'PG41628')]

In [ ]:
# dummy_meta_df

In [ ]:
#train_meta_df.head(45)

In [ ]:
# dummy_val_meta_df = train_meta_df.loc[(train_meta_df['id'] == 'PG12690') | (train_meta_df['id'] == 'PG63142')]

In [ ]:
def get_token_samples(num_samples_per_book, meta_df, tokenizer, max_length=512):
    '''Extract num_samples_per_book of max size truncated from left'''
    # Dictionary to store samples by author
    author_samples = {}
    author_attention_masks = {}

    # for each author
    for author in meta_df['author'].unique():
        author_df = meta_df.loc[meta_df['author'] == author]
        author_samples[author] = []
        author_attention_masks[author] = []

        # for each book id, get full book
        for id in author_df['id'].unique():

            try:
                book_str = get_book(id, level='text')
            except FileNotFoundError:
                print(f"{id} missing for author {author}")
                continue


            # tokenize full text (no truncation)
            # it returns a dict with input_ids & attention_mask
            tokens_dict = tokenizer(book_str, truncation=False, return_tensors="pt")
            tokens = tokens_dict["input_ids"][0]
            masks = tokens_dict["attention_mask"][0]

            # calculate total tokens needed
            total_tokens_needed = num_samples_per_book * max_length

            # If the book is too short, skip or pad as needed
            if len(tokens) < total_tokens_needed:
                # Option 1: Skip short books
                # continue

                # Option 2: Use what we have and pad the rest with repetition
                while len(tokens) < total_tokens_needed:
                    tokens = torch.cat([tokens, tokens])
                    masks = torch.cat([masks, masks])

            # Take the last total_tokens_needed tokens
            last_n_tokens = tokens[-total_tokens_needed:]
            last_n_masks = masks[-total_tokens_needed:]

            # Partition into num_samples_per_book samples
            for sample_num in range(num_samples_per_book):
                start_idx = sample_num * max_length
                end_idx = start_idx + max_length

                sample = last_n_tokens[start_idx:end_idx].tolist()
                attention_mask = last_n_masks[start_idx:end_idx].tolist()

                # Store in the author's samples list
                author_samples[author].append(sample)
                author_attention_masks[author].append(attention_mask)

    # Convert to format ready for dataset creation
    all_samples = []
    all_labels = []
    all_attention_masks = []
    label_to_id = {author: idx for idx, author in enumerate(author_samples.keys())}

    for author, samples in author_samples.items():
        label_id = label_to_id[author]
        for i,sample in enumerate(samples):
            all_samples.append(sample)
            all_attention_masks.append(author_attention_masks[author][i])
            all_labels.append(label_id)

    return all_samples, all_attention_masks, all_labels, label_to_id

In [ ]:
# dummy_meta_df

In [ ]:
num_samples_per_book = 5
max_length = 512 #DistilBert specific

In [ ]:
train_samples, train_attention_masks, train_labels, train_label_to_id = get_token_samples(num_samples_per_book,
                                                       train_meta_df,
                                                       tokenizer,
                                                       max_length=max_length)

PG3748 missing for author Verne, Jules
PG75746 missing for author Stratemeyer, Edward
PG75629 missing for author Belloc, Hilaire
PG75642 missing for author Wells, Carolyn
PG75665 missing for author Meade, L. T.
PG75765 missing for author Blanchard, Amy Ella
PG75758 missing for author Leinster, Murray
PG60311 missing for author Westerman, Percy F. (Percy Francis)
PG75760 missing for author Le Queux, William


In [ ]:
import sys

In [ ]:
print(f"train_samples: {sys.getsizeof(train_samples)} bytes")
print(f"train_attention_masks: {sys.getsizeof(train_attention_masks)} bytes")
print(f"train_labels: {sys.getsizeof(train_labels)} bytes")
print(f"train_label_to_id: {sys.getsizeof(train_label_to_id)} bytes")

train_samples: 85176 bytes
train_attention_masks: 85176 bytes
train_labels: 85176 bytes
train_label_to_id: 2272 bytes


In [ ]:
val_samples,val_attention_masks, val_labels, val_label_to_id = get_token_samples(num_samples_per_book,
                                                       val_meta_df,
                                                       tokenizer,
                                                       max_length=max_length)

In [ ]:
import pickle

In [ ]:
# save it as pkl
path = "/Users/nikitaparulekar/Desktop/MachineLearningTheory/FinalProject/saved_data/approach1/tokenized_data"



In [ ]:
# all_samples, all_attention_masks, all_labels, label_to_id = get_token_samples(3,
#                                                        dummy_meta_df,
#                                                        tokenizer,
#                                                        max_length=512)

Token indices sequence length is longer than the specified maximum sequence length for this model (65416 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
# val_samples,val_attention_mask, val_labels, val_label_to_id = get_token_samples(3,
#                                                        dummy_val_meta_df,
#                                                        tokenizer,
#                                                        max_length=512)

In [ ]:
# check sample

In [ ]:
# check sample length
# for i in range(len(all_samples)):
#     print(len(all_samples[i]))


512
512
512
512
512
512


In [ ]:
len(all_samples)

6

In [ ]:
# convert above into a Pytorch Dataset object



In [ ]:
class CustomTextDataset(Dataset):
    def __init__(self, labels, samples, attention_masks, transform=None, target_transform=None):
        self.labels = labels
        self.samples = samples
        self.attention_masks = attention_masks
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Get the token sequence and attention mask at the given index
        sample = self.samples[idx]
        attention_mask = self.attention_masks[idx]

        # Convert to tensors if not already tensors
        if not isinstance(sample, torch.Tensor):
            sample = torch.tensor(sample)
        if not isinstance(attention_mask, torch.Tensor):
            attention_mask = torch.tensor(attention_mask)

        # Get the corresponding label
        label = self.labels[idx]

        if self.transform:
            sample = self.transform(sample)
        if self.target_transform:
            label = self.target_transform(label)

        # Return in the format expected by HuggingFace Trainer
        return {
            "input_ids": sample,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset =  CustomTextDataset(
    labels=train_labels,
    samples=train_samples,
    attention_masks=train_attention_masks
)

val_dataset = CustomTextDataset(
    labels=val_labels,
    samples=val_samples,
    attention_masks=val_attention_masks)

In [ ]:
val_dataset

In [ ]:
len(train_dataset)

9555

In [ ]:
# from datasets import load_dataset

In [ ]:
# emotions = load_dataset("emotion")

In [ ]:
# emotions["train"][:2]


In [ ]:
# type(emotions["train"])

In [ ]:
## performance metrics

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average="weighted") # dont really need weightd since no class imbal in chuning and agreation methd. but could be there in othr methods due to diffrent txt length
    recall = recall_score(labels, preds, average="weighted")
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
## loading the pretrained model

In [ ]:
model_ckpt = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
os.environ["PYTORCH_ENABLE_MPS_DEVICE"] = "0"


In [ ]:
device

device(type='cpu')

In [ ]:
num_labels = train_meta_df['author'].nunique()
model = (AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels) .to(device))

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
## training the model (logged in via terminal)

In [ ]:
batch_size = 32 #CHANGE This to 64 later
logging_steps = len(train_dataset) // batch_size
model_name = f"{model_ckpt}-finetuned-author-classification"
training_args = TrainingArguments(output_dir=model_name,
                                  num_train_epochs=2,
                                  learning_rate=2e-5,
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  weight_decay=0.01,
                                  eval_strategy="epoch",
                                  disable_tqdm=False,
                                  logging_steps=logging_steps,
                                  push_to_hub=False, ## change this later
                                  log_level="error",
                                #   dataloader_pin_memory=False,     # Disable pinned memory
                                  no_cuda=True        # Explicitly disable CUDA/GPU
)

/opt/anaconda3/envs/transformers/lib/python3.10/site-packages/transformers/training_args.py:1595: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


In [ ]:
trainer = Trainer(model=model,
                  args=training_args,
                  compute_metrics=compute_metrics,
                  train_dataset=train_dataset,
                  eval_dataset=val_dataset,
                  tokenizer=tokenizer)


/var/folders/9_/jqyqn5j16938_3209cy82j2h0000gn/T/ipykernel_1574/3811405122.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 